In [2]:
import pandas as pd
import numpy as np
import re
import joblib
import os

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, f1_score, recall_score, matthews_corrcoef
from sklearn.metrics import classification_report, roc_auc_score, precision_score, confusion_matrix
from xgboost import XGBClassifier
from transformers import AutoTokenizer
import warnings
import torch
import shap
import nltk
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize


from datetime import datetime
import time

#nltk.download('punkt')
#nltk.download('punkt_tab')


"""   

Copyright (c) 2026, Michael Tchuindjang
All rights reserved.

This code was developed as part of a PhD research project in Cybersecurity and Artificial Intelligence, 
supported by a studentship at the University of the West of England (UWE Bristol).

Use of this software is permitted for academic, educational, and research purposes.  
For any commercial use or redistribution, please contact the author for permission.

Disclaimer:
In no event shall the author or UWE be liable for any claim, damages, or other liability arising from the use of this code.

Acknowledgment of the author and the research context is appreciated in any derivative work or publication.


"""

# =========================
# GLOBAL PARAMETERS
# =========================
TRAIN_DIR = "training"
os.makedirs(TRAIN_DIR, exist_ok=True)

TEST_DIR = "testing"
os.makedirs(TEST_DIR, exist_ok=True)

TESTS = ['Test_1', 
         'Test_2', 
         'Test_3', 
         'Test_4']

TRAINING_TEST = TESTS[1] # TESTS[1]: Test_2
# Tests/training files are formatted like Test_X_all_models.csv
TRAINING_TEST_FILE = TRAINING_TEST + '_all_models.csv'
TRAINING_TEST_PATH = os.path.join(TRAIN_DIR, TRAINING_TEST_FILE)


TESTING_TEST = TESTS[0] # TESTS[0]: Test_1
# Tests/training files are formatted like Test_X_all_models.csv
TESTING_TEST_FILE = TESTING_TEST + '_all_models.csv'
TESTING_TEST_PATH = os.path.join(TEST_DIR, TESTING_TEST_FILE)

CHUNKING_POOLING = "norm_mean" # change: mean / weighted / max / norm_mean
CHUNKING_OVERLAP = 40  # 0, 20, 40, 60
PHASES = ['training','testing']
TIMING_RESULTS_FILE = "timing_results.csv"

EPS = 1e-5

# Embedding models
EMB_MODEL_NAMES = ['all-MiniLM-L6-v2', 'all-mpnet-base-v2', 'all-roberta-large-v1']
EMBEDDING_MODEL_NAME = EMB_MODEL_NAMES[0]

USE_FULL_CONVERSATION = False # True = full conversation, False = last response only

USE_CHUNKING = False

CHUNK_TAG = "chunked" if USE_CHUNKING else "no_chunk"

SELECTION_METHOD = "kmeans" #"kmeans" or "hdbscan"

RUN_SIGNATURE = f"{EMBEDDING_MODEL_NAME}__{CHUNK_TAG}__full-{USE_FULL_CONVERSATION}__{TRAINING_TEST}"

EMB_MODEL_FOLDER = os.path.join(TRAIN_DIR, f"{EMBEDDING_MODEL_NAME}")
os.makedirs(EMB_MODEL_FOLDER, exist_ok=True)

#REFUSAL_FILE = os.path.join(EMB_MODEL_FOLDER, f"{RUN_SIGNATURE}__model_agnostic_{SELECTION_METHOD}_refusal_set.csv")

MODEL_PATH = os.path.join(EMB_MODEL_FOLDER, f"{RUN_SIGNATURE}__xgb_{SELECTION_METHOD}_refusal_model.pkl")
#REFUSAL_BANK_PATH = os.path.join(EMB_MODEL_FOLDER, f"{RUN_SIGNATURE}__{SELECTION_METHOD}_refusal_bank.pkl")

REFUSAL_BANK_PATH = os.path.join(
    EMB_MODEL_FOLDER,
    f"{RUN_SIGNATURE}__model_agnostic_{SELECTION_METHOD}_refusal_full_snapshot.pkl"
)

TRAINING_METRIC_FILE_PREFIX = os.path.join(EMB_MODEL_FOLDER, f"training_metrics_{RUN_SIGNATURE}__{SELECTION_METHOD}__")

SHAP_PLOT_FILE = os.path.join(EMB_MODEL_FOLDER, f"shap_summary_{RUN_SIGNATURE}__{SELECTION_METHOD}__plot.png")


embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/" + EMBEDDING_MODEL_NAME)

def get_final_response(row):
    return row.get(f"output_turn_{row['turn_depth']}", "")

def get_full_conversation(row):
    conv = []
    for i in range(1, row["turn_depth"] + 1):
        u = row.get(f"turn_{i}", "")
        a = row.get(f"output_turn_{i}", "")
        if pd.notna(u) and u.strip():
            conv.append(f"USER: {u}")
        if pd.notna(a) and a.strip():
            conv.append(f"ASSISTANT: {a}")
    return "\n".join(conv)


def save_experiment_timing(
    file_path=TIMING_RESULTS_FILE,
    model=EMBEDDING_MODEL_NAME,
    phase="",
    test="",
    chunking=USE_CHUNKING,
    pooling=CHUNKING_POOLING,
    overlap=CHUNKING_OVERLAP,
    text_embed_time=0.0,
    feature_time=0.0,
    training_time=0.0,
    testing_time=0.0,
    total_time=0.0
):

    # =========================
    # CONVERT SEC → MS
    # =========================
    text_embed_time *= 1000
    feature_time *= 1000
    training_time *= 1000
    testing_time *= 1000
    total_time *= 1000
    
    now = datetime.now()
    
    # New row as DataFrame
    new_row = pd.DataFrame([{
        "model": model,
        "phase": phase,
        "test": test,
        "chunking": chunking,
        "pooling": pooling,
        "overlap": overlap,
        "text_embed_time_ms": text_embed_time,
        "feature_time_ms": feature_time,
        "training_time_ms": training_time,
        "testing_time_ms": testing_time,
        "total_time_ms": total_time,
        "timestamp": now.strftime("%Y-%m-%d %H:%M:%S")
    }])

    # Append or create file
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df = pd.concat([df, new_row], ignore_index=True)
    else:
        df = new_row

    df.to_csv(file_path, index=False)

    print(f"⏱️ Experiment timing saved to {file_path}")

# =========================
# CHUNKING ENCODER
# =========================
def embed_no_chunk(texts, model, tokenizer):
    """
    Encode list of texts using standard truncated embedding.
    
    Args:
        texts (list[str])
        model (SentenceTransformer)

    Returns:
        np.ndarray: shape (N, D)
    """
    return model.encode(texts, convert_to_numpy=True, batch_size=64, show_progress_bar=True)


def chunk_text_token_level(text, tokenizer, max_tokens=256, overlap=CHUNKING_OVERLAP):
    """
    Token-consistent chunking using model tokenizer.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]

        chunk = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk.strip())

        start += max_tokens - overlap

    return chunks

# =========================================================
# 3. POOLING STRATEGIES (UNBIASED OPTIONS)
# =========================================================

def mean_pool(embeddings):
    return np.mean(embeddings, axis=0)

def weighted_mean_pool(embeddings, chunks, tokenizer):
    weights = np.array([
        len(tokenizer.encode(c, add_special_tokens=False))
        for c in chunks
    ])
    return np.average(embeddings, axis=0, weights=weights)

def max_pool(embeddings):
    return np.max(embeddings, axis=0)

def normalized_mean_pool(embeddings):
    emb = normalize(embeddings, axis=1)
    pooled = np.mean(emb, axis=0)
    return normalize(pooled.reshape(1, -1))[0]

# POOLING_METHOD is "weighted" by default  # change: mean / weighted / max / norm_mean
def embed_chunk(texts, model, tokenizer, pooling=CHUNKING_POOLING, batch_size=64):
    all_embeddings = []
    chunk_counts = []
    all_chunks = []

    # -----------------------------
    # Flatten chunks
    # -----------------------------
    for text in texts:
        chunks = chunk_text_token_level(text, tokenizer)

        if len(chunks) == 0:
            chunks = [""]

        chunk_counts.append(len(chunks))
        all_chunks.extend(chunks)

    # -----------------------------
    # Encode chunks
    # -----------------------------
    chunk_embeddings = model.encode(
        all_chunks,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    # -----------------------------
    # Pool per document
    # -----------------------------
    idx = 0

    for i, count in enumerate(chunk_counts):
        doc_chunks = chunk_embeddings[idx:idx + count]
        idx += count

        if pooling == "mean":
            doc_emb = mean_pool(doc_chunks)

        elif pooling == "weighted":
            doc_emb = weighted_mean_pool(doc_chunks, all_chunks[:count], tokenizer)

        elif pooling == "max":
            doc_emb = max_pool(doc_chunks)

        elif pooling == "norm_mean":
            doc_emb = normalized_mean_pool(doc_chunks)

        else:
            raise ValueError("Unknown pooling method")

        all_embeddings.append(doc_emb)

    return np.array(all_embeddings)


# =========================
# ENCODING WRAPPER
# =========================
def encode_texts(texts, model, tokenizer):
    if USE_CHUNKING:
        embeddings = embed_chunk(texts, model, tokenizer)
        if isinstance(embeddings, torch.Tensor):
            embeddings = embeddings.detach().cpu().numpy()
        return embeddings
    return embed_no_chunk(texts, model, tokenizer)

# =========================
# CONFIG
# =========================
FEATURE_GROUPS = {

    # =========================================================
    # 1. LOCAL MANIFOLD (cluster-level alignment + structure)
    # =========================================================
    "local_manifold": [
        "sim_max_local",
        "sim_entropy_local"
    ],

    # =========================================================
    # 2. GLOBAL MANIFOLD (refusal set alignment)
    # =========================================================
    "global_manifold": [
        "sim_max_global",
        "sim_entropy_global",
        "softmax_refusal_mass"
    ],

    # =========================================================
    # 3. DISTRIBUTION GEOMETRY (statistical deviation)
    # =========================================================
    "geometry": [
        "maha_refusal"
    ],

    # =========================================================
    # 4. CROSS-SCALE CONSISTENCY (key jailbreak signal)
    # =========================================================
    "cross_scale": [
        "cross_ratio"
    ],

    # =========================================================
    # 5. STRUCTURAL SIGNALS (length bias control)
    # =========================================================
    "structure": [
        "length_log"
    ]
}

# =========================
# FEATURE SET GENERATION
# =========================
def get_feature_sets():
    all_groups = list(FEATURE_GROUPS.keys())
    configs = {}

    # Full model
    configs["all_features"] = sum(FEATURE_GROUPS.values(), [])

    # Leave-one-group-out
    for g in all_groups:
        remaining = [grp for grp in all_groups if grp != g]
        features = []
        for r in remaining:
            features.extend(FEATURE_GROUPS[r])
        configs[f"minus_{g}"] = features

    return configs

def compute_cov_inv(embeddings, eps=EPS):
    cov = np.cov(embeddings, rowvar=False)
    cov += np.eye(cov.shape[0]) * eps
    return np.linalg.inv(cov)


def mahalanobis_distance(x, mean, cov_inv):
    diff = x - mean
    return np.sqrt(diff @ cov_inv @ diff.T)

# =========================
# FEATURE CONSTRUCTION
# =========================
def build_feature_df(texts, text_embeddings, refusal_embeddings, centroids):

    rows = []
    semantic_scores = []

    # =========================================================
    # NORMALIZATION
    # =========================================================
    text_embeddings = normalize(text_embeddings, axis=1)
    refusal_embeddings = normalize(refusal_embeddings, axis=1)
    centroids = normalize(centroids, axis=1)

    # =========================================================
    # GLOBAL STATISTICS (for Mahalanobis)
    # =========================================================
    refusal_mean = refusal_embeddings.mean(axis=0)
    refusal_cov_inv = compute_cov_inv(refusal_embeddings)

    for i, emb in enumerate(text_embeddings):

        row = {}

        # =========================================================
        # LOCAL MANIFOLD (semantic centroids)
        # =========================================================
        sims_local = cosine_similarity([emb], centroids)[0]

        sim_max_local = sims_local.max()
        p_local = sims_local - sims_local.min()
        p_local = p_local / (p_local.sum() + EPS)
        sim_entropy_local = -np.sum(p_local * np.log(p_local + EPS))

        # =========================================================
        # GLOBAL MANIFOLD (refusal embeddings)
        # =========================================================
        sims_global = cosine_similarity([emb], refusal_embeddings)[0]

        sim_max_global = sims_global.max()
        p_global = sims_global - sims_global.min()
        p_global = p_global / (p_global.sum() + EPS)
        sim_entropy_global = -np.sum(p_global * np.log(p_global + EPS))

        softmax_refusal_mass = np.log(np.sum(np.exp(sims_global + EPS)))

        # =========================================================
        # GEOMETRY (Mahalanobis distance to refusal distribution)
        # =========================================================
        maha_refusal = mahalanobis_distance(
            emb, refusal_mean, refusal_cov_inv
        )

        # =========================================================
        # CROSS-SCALE GEOMETRY (dominance ratio)
        # =========================================================
        cross_ratio = (sim_max_local + EPS) / (sim_max_global + EPS)

        # =========================================================
        # STRUCTURE (length prior)
        # =========================================================
        length_log = np.log1p(len(texts[i].split()))

        # =========================================================
        # FINAL FEATURE VECTOR
        # =========================================================
        row.update({

            # Local manifold
            "sim_max_local": sim_max_local,
            "sim_entropy_local": sim_entropy_local,

            # Global manifold
            "sim_max_global": sim_max_global,
            "sim_entropy_global": sim_entropy_global,
            "softmax_refusal_mass": softmax_refusal_mass,

            # Geometry
            "maha_refusal": maha_refusal,

            # Cross-scale
            "cross_ratio": cross_ratio,

            # Structure
            "length_log": length_log
        })
        
        semantic_scores.append(max(sim_max_local, sim_max_global))
        rows.append(row)

    return pd.DataFrame(rows), semantic_scores

    

def get_ablation_configs(mode="leave_one_out"):
    groups = list(FEATURE_GROUPS.keys())
    configs = {}

    # FULL
    configs["all_features"] = sum(FEATURE_GROUPS.values(), [])

    if mode == "leave_one_out":
        for g in groups:
            remaining = [grp for grp in groups if grp != g]
            feats = []
            for r in remaining:
                feats.extend(FEATURE_GROUPS[r])
            configs[f"minus_{g}"] = feats

    elif mode == "single_group":
        for g in groups:
            configs[f"only_{g}"] = FEATURE_GROUPS[g]

    elif mode == "forward":
        current = []
        for g in groups:
            current += FEATURE_GROUPS[g]
            configs[f"forward_{g}"] = current.copy()

    return configs


from sklearn.metrics import (
    precision_score,
    confusion_matrix
)

def train_and_evaluate(
    X_train,
    y_train,
    X_test,
    y_test,
    seed=42
):
    """
    Train RefusalGuard on X_train/y_train and evaluate on X_test/y_test.
    Used by both train_eval_once() and run_generalization_experiment().
    """

    pos_weight = (len(y_train) - np.sum(y_train)) / np.sum(y_train)

    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=pos_weight,
        max_delta_step=1,
        eval_metric="logloss",
        random_state=seed
    )

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]

    # -----------------------
    # Threshold optimization
    # -----------------------
    best_score = -np.inf
    best_t = 0.5

    for t in np.arange(0.10, 0.90, 0.01):

        preds = (probs >= t).astype(int)

        recall = recall_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        mcc = matthews_corrcoef(y_test, preds)

        # Recall is the dominant signal (catch most true jailbreaks), followed by MCC and F1 as a secondary balancing role
        #score = 0.2*kappa + 0.4*f1 + 0.4*recall
        score = 0.5 * recall + 0.3 * mcc + 0.2 * f1

        if score > best_score:
            best_score = score
            best_t = t

    preds = (probs >= best_t).astype(int)

    precision = precision_score(y_test, preds)

    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

    fpr = fp / (fp + tn)

    return {
        "model": model,
        "threshold": best_t,
        "predictions": preds,
        "probabilities": probs,
        "f1": f1_score(y_test, preds),
        "precision": precision,
        "recall": recall_score(y_test, preds),
        "mcc": matthews_corrcoef(y_test, preds),
        "kappa": cohen_kappa_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs),
        "fpr": fpr
    }


def train_eval_once(feature_df, y, feature_cols, seed=42):

    X = feature_df[feature_cols].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return train_and_evaluate(
        X_train,
        y_train,
        X_test,
        y_test,
        seed
    )


def run_generalization_experiment(
    train_df,
    test_df,
    train_name,
    test_name,
    setting,
    embedder,
    tokenizer
):
    """
    Train model embedding on train_df and evaluate on test_df
    under cross-benchmark or attack-disjoint settings.
    """

    # ----------------------------------------------------------
    # LOAD REFUSAL SNAPSHOT
    # ----------------------------------------------------------
    snapshot = joblib.load(REFUSAL_BANK_PATH)

    refusals = snapshot["refusal_embeddings"]
    centroids = snapshot["cluster_centroids"]

    # ----------------------------------------------------------
    # TEXT EXTRACTION
    # ----------------------------------------------------------
    if USE_FULL_CONVERSATION:

        train_texts = train_df.apply(
            get_full_conversation,
            axis=1
        ).tolist()

        test_texts = test_df.apply(
            get_full_conversation,
            axis=1
        ).tolist()

    else:

        train_texts = train_df.apply(
            get_final_response,
            axis=1
        ).tolist()

        test_texts = test_df.apply(
            get_final_response,
            axis=1
        ).tolist()


    # ----------------------------------------------------------
    # EMBEDDINGS
    # ----------------------------------------------------------
    train_embeddings = encode_texts(
        train_texts,
        embedder,
        tokenizer
    )

    test_embeddings = encode_texts(
        test_texts,
        embedder,
        tokenizer
    )


    if isinstance(train_embeddings, torch.Tensor):
        train_embeddings = train_embeddings.detach().cpu().numpy()

    if isinstance(test_embeddings, torch.Tensor):
        test_embeddings = test_embeddings.detach().cpu().numpy()


    # ----------------------------------------------------------
    # FEATURES
    # ----------------------------------------------------------
    train_features, _ = build_feature_df(
        train_texts,
        train_embeddings,
        refusals,
        centroids
    )

    test_features, _ = build_feature_df(
        test_texts,
        test_embeddings,
        refusals,
        centroids
    )


    y_train = train_df["human_ensemble_judge"].values
    y_test = test_df["human_ensemble_judge"].values


    feature_cols = sum(
        FEATURE_GROUPS.values(),
        []
    )


    X_train = train_features[feature_cols].values
    X_test = test_features[feature_cols].values


    # ----------------------------------------------------------
    # TRAIN + EVALUATE
    # ----------------------------------------------------------
    result = train_and_evaluate(
        X_train,
        y_train,
        X_test,
        y_test
    )


    # ----------------------------------------------------------
    # RETURN TABLE ROW
    # ----------------------------------------------------------
    return {
        "Setting": setting,
        "Train Data": train_name,
        "Test Data": test_name,
        "Method": "RefusalGuard",
        "F1": result["f1"],
        "Precision": result["precision"],
        "Recall": result["recall"],
        "FPR": result["fpr"]
    }


def train_model(mode="leave_one_out"):

    start_total = time.perf_counter()

    print(f"Loading data from {TRAINING_TEST_PATH}...")
    df = pd.read_csv(TRAINING_TEST_PATH)
    y = df["human_ensemble_judge"].values

    # -------------------------
    # TEXT
    # -------------------------
    if USE_FULL_CONVERSATION:
        texts = df.apply(get_full_conversation, axis=1).tolist()
    else:
        texts = df.apply(get_final_response, axis=1).tolist()

    # -------------------------
    # REFUSAL SNAPSHOT
    # -------------------------
    snapshot = joblib.load(REFUSAL_BANK_PATH)
    centroids = snapshot["cluster_centroids"]
    refusals = snapshot["refusal_embeddings"]

    # -------------------------
    # TEXT EMBEDDINGS
    # -------------------------
    print(f"🔄 Encoding training texts with {EMBEDDING_MODEL_NAME}...")
    t2 = time.perf_counter()
    text_embeddings = encode_texts(texts, embedder, tokenizer)
    if isinstance(text_embeddings, torch.Tensor):
        text_embeddings = text_embeddings.detach().cpu().numpy()

    # =====================================================
    # ⏱️ TEXT EMBEDDING TIME
    # =====================================================
    t3 = time.perf_counter()
    text_embed_time = t3 - t2

    # -------------------------
    # FEATURES (BUILT ONCE)
    # -------------------------
    print("⚙️ Building features...")
    t4 = time.perf_counter()
    feature_df, semantic_scores= build_feature_df(
        texts,
        text_embeddings,
        refusals,
        centroids
    )


    # =====================================================
    # ⏱️ FEATURE BUILDING TIME
    # =====================================================
    t5 = time.perf_counter()
    feature_time = t5 - t4

    # -------------------------
    # CONFIGS
    # -------------------------
    configs = get_ablation_configs(mode=mode)

    results = []

    # TRACK BEST
    best_model = None
    best_features = None
    best_threshold = None
    best_score = -np.inf

    print(f"\n🚀 Running ablation: {mode}")

    train_start = time.perf_counter()

    for name, feature_cols in configs.items():

        print(f"\n{name}")
        print("Features:", feature_cols)

        result = train_eval_once(feature_df, y, feature_cols)

        # same scoring used in training
        # Recall is the dominant signal (catch most true jailbreaks), followed by MCC and F1 as a secondary balancing role
        #score = 0.2 * result["kappa"] + 0.4 * result["f1"] + 0.4 * result["recall"]
        score = 0.5 * result["recall"] + 0.3 * result["mcc"] + 0.2 * result["f1"]
        
        results.append({
            "config": name,
            "n_features": len(feature_cols),
            "features": ",".join(feature_cols),
            "f1": result["f1"],
            "recall": result["recall"],
            "mcc": result["mcc"],
            "kappa": result["kappa"],
            "roc_auc": result["roc_auc"],
            "threshold": result["threshold"],
            "score": score
        })

        # UPDATE BEST MODEL
        if score > best_score:
            best_score = score
            best_model = result["model"]
            best_features = feature_cols
            best_threshold = result["threshold"]

    # =====================================================
    # ⏱️ MODEL TRAINING TIME
    # =====================================================

    train_end = time.perf_counter()
    model_train_time = train_end - train_start

    # -------------------------
    # RESULTS TABLE
    # -------------------------
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="score", ascending=False)
    output_file=TRAINING_METRIC_FILE_PREFIX + mode + ".csv"
    results_df.to_csv(output_file, index=False)

    print("\n📊 FINAL RANKING:")
    print(results_df)

    # -------------------------
    # SAVE BEST MODEL ONLY
    # -------------------------
    print("\nBest configuration:")
    #print("Features:", best_features)
    #print("Score:", best_score)
    best_row = results_df.iloc[0]
    print(best_row.to_string())

    joblib.dump({
        "model": best_model,
        "threshold": best_threshold,
        "feature_cols": best_features
    }, MODEL_PATH)

    
    print("\n🎯 Computing SHAP for best model...")

    X_best = feature_df[best_features].values

    X_train, X_test, y_train, y_test = train_test_split(
        X_best, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)

    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=best_features,
        show=False
    )

    # Save figure
    plt.savefig(f"{SHAP_PLOT_FILE}", bbox_inches='tight', dpi=300)
    # Optional: display after saving
    plt.show()

    # =====================================================
    # ⏱️ TOTAL TIME
    # =====================================================

    end_total = time.perf_counter()
    total_time = end_total - start_total

    # We only add pooling and overlap if chunking is True
    if USE_CHUNKING:
        pool = CHUNKING_POOLING
        overlp = CHUNKING_OVERLAP
    else:
        pool = None
        overlp = None
    
    save_experiment_timing(
    file_path=TIMING_RESULTS_FILE,
    model=EMBEDDING_MODEL_NAME,
    phase=PHASES[0], #training phase
    test=TRAINING_TEST,
    chunking=USE_CHUNKING,
    pooling=pool,
    overlap=overlp,
    text_embed_time=text_embed_time,
    feature_time=feature_time,
    training_time=model_train_time,
    testing_time=0.0,
    total_time=total_time)

    print("\n✅ Training + ablation completed, and best model saved.")


def load_and_predict(
    new_file,
    model_filter=None,
    tense_filter=None,
    turn_depth_filter=None,
    source_filter=None,
    row_start=None,
    row_end=None
):

    start_total = time.perf_counter()
    print("🔄 Loading classifier model + refusal bank...")

    # -------------------------
    # LOAD MODEL BUNDLE
    # -------------------------
    bundle = joblib.load(MODEL_PATH)

    model = bundle["model"]
    best_t = bundle["threshold"]
    feature_cols = bundle["feature_cols"]   #  AUTO LOAD FEATURES

    # -------------------------
    # LOAD REFUSAL BANK
    # -------------------------
    snapshot = joblib.load(REFUSAL_BANK_PATH)
    centroids = snapshot["cluster_centroids"]
    refusals = snapshot["refusal_embeddings"]

    print(f"Loading data from {new_file}...")
    df = pd.read_csv(new_file)

    # -------------------------
    # OPTIONAL FILTERS
    # -------------------------
    if model_filter:
        df = df[df["model_name"] == model_filter]
    if tense_filter:
        df = df[df["tense"] == tense_filter]
    if turn_depth_filter:
        df = df[df["turn_depth"] == turn_depth_filter]
    if source_filter:
        df = df[df["source"] == source_filter]

    if row_start is not None or row_end is not None:
        df = df.iloc[row_start:row_end]

    df = df.reset_index(drop=True)

    # -------------------------
    # TEXT SELECTION
    # -------------------------
    if USE_FULL_CONVERSATION:
        texts = df.apply(get_full_conversation, axis=1).tolist()
    else:
        texts = df.apply(get_final_response, axis=1).tolist()


    t0 = time.perf_counter()
    print(f"🔄 Encoding new texts with {EMBEDDING_MODEL_NAME}...")
    text_embeddings = encode_texts(texts, embedder, tokenizer)

    if isinstance(text_embeddings, torch.Tensor):
        text_embeddings = text_embeddings.detach().cpu().numpy()

    # =====================================================
    # ⏱️ TEXT EMBEDDING TIME
    # =====================================================
    t1 = time.perf_counter()
    text_embed_time = t1 - t0

    t2 = time.perf_counter()

    # -------------------------
    # BUILD FULL FEATURE SET
    # -------------------------
    feature_df, semantic_scores= build_feature_df(
        texts,
        text_embeddings,
        refusals,
        centroids
    )

    # -------------------------
    # SELECT CORRECT FEATURES
    # -------------------------
    missing = [f for f in feature_cols if f not in feature_df.columns]
    if missing:
        raise ValueError(f"Missing features at inference: {missing}")

    X = feature_df[feature_cols].values

    # =====================================================
    # ⏱️ FEATURE ENGINEERING TIME
    # =====================================================

    t3 = time.perf_counter()
    feature_time = t3 - t2

    t4 = time.perf_counter()

    # -------------------------
    # PREDICTION
    # -------------------------
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= best_t).astype(int)
    conf = np.abs(probs - best_t)

    # =====================================================
    # ⏱️ PREDICTION TIME
    # =====================================================

    t5 = time.perf_counter()
    prediction_time = t5 - t4

    # -------------------------
    # SAVE OUTPUT
    # -------------------------
    tag = f"{EMBEDDING_MODEL_NAME}[{CHUNK_TAG}]"

    df[f"semantic_score_{tag}"] = semantic_scores
    df[f"prob_jailbreak_{tag}"] = probs
    df[f"prediction_{tag}_judge"] = preds
    df[f"confidence_{tag}"] = conf

    df.to_csv(new_file, index=False)

    print(f"✅ Saved → {new_file}")

    # -------------------------
    # TOTAL TIME
    # -------------------------
    end_total = time.perf_counter()
    total_time = end_total - start_total


    # We only add pooling and overlap if chunking is True
    if USE_CHUNKING:
        pool = CHUNKING_POOLING
        overlp = CHUNKING_OVERLAP
    else:
        pool = None
        overlp = None
    
    save_experiment_timing(
    file_path=TIMING_RESULTS_FILE,
    model=EMBEDDING_MODEL_NAME,
    phase=PHASES[1], #testing phase
    test=TESTING_TEST,    
    chunking=USE_CHUNKING,
    pooling=pool,
    overlap=overlp,
    text_embed_time=text_embed_time,
    feature_time=feature_time,
    training_time=0.0,
    testing_time=prediction_time,
    total_time=total_time)


def run_full_experiments(
    model_names,
    chunking_options=(False, True),
    ablation_mode="leave_one_out"
):

    global EMBEDDING_MODEL_NAME, USE_CHUNKING, CHUNK_TAG
    global MODEL_PATH, REFUSAL_BANK_PATH
    global TRAINING_METRIC_FILE_PREFIX, SHAP_PLOT_FILE
    global EMB_MODEL_FOLDER, RUN_SIGNATURE
    global embedder, tokenizer

    for model_name in model_names:
        for chunking in chunking_options:

            print("\n" + "="*60)
            print(f"🚀 Running: model={model_name} | chunking={chunking}")
            print("="*60)

            # -------------------------
            # SET GLOBAL CONFIG
            # -------------------------
            EMBEDDING_MODEL_NAME = model_name
            USE_CHUNKING = chunking

            CHUNK_TAG = "chunked" if chunking else "no_chunk"

            RUN_SIGNATURE = f"{EMBEDDING_MODEL_NAME}__{CHUNK_TAG}__full-{USE_FULL_CONVERSATION}__{TRAINING_TEST}"

            EMB_MODEL_FOLDER = os.path.join(TRAIN_DIR, f"{EMBEDDING_MODEL_NAME}")
            os.makedirs(EMB_MODEL_FOLDER, exist_ok=True)
            
            MODEL_PATH = os.path.join(EMB_MODEL_FOLDER, f"{RUN_SIGNATURE}__xgb_{SELECTION_METHOD}_refusal_model.pkl")
            
            REFUSAL_BANK_PATH = os.path.join(
                EMB_MODEL_FOLDER,
                f"{RUN_SIGNATURE}__model_agnostic_{SELECTION_METHOD}_refusal_full_snapshot.pkl"
            )
            
            TRAINING_METRIC_FILE_PREFIX = os.path.join(EMB_MODEL_FOLDER, f"training_metrics_{RUN_SIGNATURE}__{SELECTION_METHOD}__")
            
            SHAP_PLOT_FILE = os.path.join(EMB_MODEL_FOLDER, f"shap_summary_{RUN_SIGNATURE}__{SELECTION_METHOD}__plot.png")

            # -------------------------
            # LOAD MODEL + TOKENIZER
            # -------------------------
            print(f"🔄 Loading embedding model: {model_name}")
            embedder = SentenceTransformer(model_name)
            tokenizer = AutoTokenizer.from_pretrained(
                "sentence-transformers/" + model_name
            )

            # -------------------------
            # TRAIN
            # -------------------------
            train_model(mode=ablation_mode)

            # -------------------------
            # TEST / PREDICT
            # -------------------------
            load_and_predict(TESTING_TEST_PATH)

            print(f"✅ Done: {model_name} | chunking={chunking}")


def run_generalization_experiments(
    model_names,
    chunking_options=(False, True),
    ablation_mode="leave_one_out"
):

    global EMBEDDING_MODEL_NAME, USE_CHUNKING, CHUNK_TAG
    global MODEL_PATH, REFUSAL_BANK_PATH
    global TRAINING_METRIC_FILE_PREFIX, SHAP_PLOT_FILE
    global EMB_MODEL_FOLDER, RUN_SIGNATURE
    global embedder, tokenizer

    all_results = []

    for model_name in model_names:
        for chunking in chunking_options:

            print("\n" + "=" * 60)
            print(f"Running Generalization: {model_name} | chunking={chunking}")
            print("=" * 60)

            # -------------------------------------------------
            # SAME CONFIGURATION AS run_full_experiments()
            # -------------------------------------------------
            EMBEDDING_MODEL_NAME = model_name
            USE_CHUNKING = chunking

            CHUNK_TAG = "chunked" if chunking else "no_chunk"

            RUN_SIGNATURE = (
                f"{EMBEDDING_MODEL_NAME}"
                f"__{CHUNK_TAG}"
                f"__full-{USE_FULL_CONVERSATION}"
                f"__{TRAINING_TEST}"
            )

            EMB_MODEL_FOLDER = os.path.join(TRAIN_DIR, EMBEDDING_MODEL_NAME)
            os.makedirs(EMB_MODEL_FOLDER, exist_ok=True)

            MODEL_PATH = os.path.join(
                EMB_MODEL_FOLDER,
                f"{RUN_SIGNATURE}__xgb_{SELECTION_METHOD}_refusal_model.pkl"
            )

            REFUSAL_BANK_PATH = os.path.join(
                EMB_MODEL_FOLDER,
                f"{RUN_SIGNATURE}__model_agnostic_{SELECTION_METHOD}_refusal_full_snapshot.pkl"
            )

            TRAINING_METRIC_FILE_PREFIX = os.path.join(
                EMB_MODEL_FOLDER,
                f"training_metrics_{RUN_SIGNATURE}__{SELECTION_METHOD}__"
            )

            SHAP_PLOT_FILE = os.path.join(
                EMB_MODEL_FOLDER,
                f"shap_summary_{RUN_SIGNATURE}__{SELECTION_METHOD}__plot.png"
            )

            # -------------------------------------------------
            # LOAD EMBEDDING MODEL
            # -------------------------------------------------
            embedder = SentenceTransformer(model_name)
            tokenizer = AutoTokenizer.from_pretrained(
                "sentence-transformers/" + model_name
            )

            # -------------------------------------------------
            # LOAD DATA
            # -------------------------------------------------
            df = pd.read_csv(TRAINING_TEST_PATH)

            benchmarks = sorted(df["source"].dropna().unique())
            attacks = sorted(df["attack_name"].dropna().unique())
            tenses = sorted(df["tense"].dropna().unique())
            
            print("Benchmarks:", benchmarks)
            print("Attack types:", attacks)
            print("Tenses:", tenses)

            # -------------------------------------------------
            # CROSS-BENCHMARK
            # -------------------------------------------------
            for test_source in benchmarks:

                train_df = df[df["source"] != test_source]
                test_df = df[df["source"] == test_source]

                if len(train_df) == 0 or len(test_df) == 0:
                    continue

                result = run_generalization_experiment(
                    train_df=train_df,
                    test_df=test_df,
                    train_name=" + ".join(sorted(train_df["source"].unique())),
                    test_name=test_source,
                    setting="Cross-benchmark",
                    embedder=embedder,
                    tokenizer=tokenizer
                )

                result["Embedding Model"] = model_name
                result["Chunking"] = chunking

                all_results.append(result)

            # -------------------------------------------------
            # ATTACK-DISJOINT
            # -------------------------------------------------
            for test_attack in attacks:

                train_df = df[df["attack_name"] != test_attack]
                test_df = df[df["attack_name"] == test_attack]

                if len(train_df) == 0 or len(test_df) == 0:
                    continue

                result = run_generalization_experiment(
                    train_df=train_df,
                    test_df=test_df,
                    train_name=" + ".join(sorted(train_df["attack_name"].unique())),
                    test_name=test_attack,
                    setting="Attack-disjoint",
                    embedder=embedder,
                    tokenizer=tokenizer
                )

                result["Embedding Model"] = model_name
                result["Chunking"] = chunking

                all_results.append(result)

            # -------------------------------------------------
            # TENSE-DISJOINT
            # -------------------------------------------------
            for test_tense in tenses:
            
                train_df = df[df["tense"] != test_tense]
                test_df = df[df["tense"] == test_tense]
            
                if len(train_df) == 0 or len(test_df) == 0:
                    continue
            
                result = run_generalization_experiment(
                    train_df=train_df,
                    test_df=test_df,
                    train_name=" + ".join(sorted(train_df["tense"].astype(str).unique())),
                    test_name=test_tense,
                    setting="Tense-disjoint",
                    embedder=embedder,
                    tokenizer=tokenizer
                )
            
                result["Embedding Model"] = model_name
                result["Chunking"] = chunking
            
                all_results.append(result)
            

    results_df = pd.DataFrame(all_results)

    output_file = os.path.join(
        TRAIN_DIR,
        f"generalization_results_{TRAINING_TEST}.csv"
    )

    results_df.to_csv(output_file, index=False)

    print(results_df)

    print(f"\n✅ Saved generalization results to {output_file}")

    return results_df

# =========================
# EXAMPLE USAGE
# =========================
#train_model(mode="forward") # "leave_one_out", "single_group", "forward"
#load_and_predict(TESTING_TEST_PATH)
"""load_and_predict(
     "Test_1_all_models.csv",
     feature_cols,
     model_filter="gpt-4o-mini",
     tense_filter="past",
     turn_depth_filter=3,
     source_filter="AdvBench"
     row_start=0,
     row_end=10
)"""

"""run_full_experiments(
    model_names=EMB_MODEL_NAMES,
    chunking_options=[False, True],  # try both
    ablation_mode="leave_one_out"  # or "leave_one_out"
)"""

run_generalization_experiments(
    model_names=EMB_MODEL_NAMES,
    #model_names=['all-MiniLM-L6-v2'],
    chunking_options=[False, True],
    ablation_mode="leave_one_out"
)


Running Generalization: all-MiniLM-L6-v2 | chunking=False
Benchmarks: ['AdvBench', 'HarmBench', 'Original']
Attack types: ['crescendo', 'mirage', 'tempest']
Tenses: ['past', 'present']


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/43 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

Batches:   0%|          | 0/122 [00:00<?, ?it/s]

            Setting            Train Data  Test Data        Method        F1  \
0   Cross-benchmark  HarmBench + Original   AdvBench  RefusalGuard  0.697174   
1   Cross-benchmark   AdvBench + Original  HarmBench  RefusalGuard  0.454695   
2   Cross-benchmark  AdvBench + HarmBench   Original  RefusalGuard  0.718611   
3   Attack-disjoint      mirage + tempest  crescendo  RefusalGuard  0.694006   
4   Attack-disjoint   crescendo + tempest     mirage  RefusalGuard  0.654873   
5   Attack-disjoint    crescendo + mirage    tempest  RefusalGuard  0.649425   
6    Tense-disjoint               present       past  RefusalGuard  0.750000   
7    Tense-disjoint                  past    present  RefusalGuard  0.679155   
8   Cross-benchmark  HarmBench + Original   AdvBench  RefusalGuard  0.692513   
9   Cross-benchmark   AdvBench + Original  HarmBench  RefusalGuard  0.448780   
10  Cross-benchmark  AdvBench + HarmBench   Original  RefusalGuard  0.735632   
11  Attack-disjoint      mirage + tempes

,Setting,Train Data,Test Data,Method,F1,Precision,Recall,FPR,Embedding Model,Chunking
0,Cross-benchmark,HarmBench + Original,AdvBench,RefusalGuard,0.697174,0.537344,0.992337,0.527187,all-MiniLM-L6-v2,False
1,Cross-benchmark,AdvBench + Original,HarmBench,RefusalGuard,0.454695,0.294872,0.992806,0.470756,all-MiniLM-L6-v2,False
2,Cross-benchmark,AdvBench + HarmBench,Original,RefusalGuard,0.718611,0.729656,0.707895,0.335202,all-MiniLM-L6-v2,False
3,Attack-disjoint,mirage + tempest,crescendo,RefusalGuard,0.694006,0.536585,0.982143,0.669014,all-MiniLM-L6-v2,False
4,Attack-disjoint,crescendo + tempest,mirage,RefusalGuard,0.654873,0.602041,0.717871,0.360069,all-MiniLM-L6-v2,False
5,Attack-disjoint,crescendo + mirage,tempest,RefusalGuard,0.649425,0.480851,1.000000,0.865248,all-MiniLM-L6-v2,False
6,Tense-disjoint,present,past,RefusalGuard,0.750000,0.661395,0.866017,0.517781,all-MiniLM-L6-v2,False
7,Tense-disjoint,past,present,RefusalGuard,0.679155,0.518709,0.983310,0.499619,all-MiniLM-L6-v2,False
8,Cross-benchmark,HarmBench + Original,AdvBench,RefusalGuard,0.692513,0.531828,0.992337,0.539007,all-MiniLM-L6-v2,True
9,Cross-benchmark,AdvBench + Original,HarmBench,RefusalGuard,0.448780,0.289916,0.992806,0.482168,all-MiniLM-L6-v2,True
